# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with the APIs for Anthropic and Google, as well as OpenAI.

## Setting up your keys

If you haven't done so already, you could now create API keys for Anthropic and Google in addition to OpenAI.

**Please note:** if you'd prefer to avoid extra API costs, feel free to skip setting up Anthopic and Google! You can see me do it, and focus on OpenAI for the course. You could also substitute Anthropic and/or Google for Ollama, using the exercise you did in week 1.

For OpenAI, visit https://openai.com/api/  
For Anthropic, visit https://console.anthropic.com/  
For Google, visit https://ai.google.dev/gemini-api  

When you get your API keys, you need to set them as environment variables by adding them to your `.env` file.

```
OPENAI_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
GOOGLE_API_KEY=xxxx
```

Afterwards, you may need to restart the Jupyter Lab Kernel (the Python process that sits behind this notebook) via the Kernel menu, and then rerun the cells from the top.

In [31]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
from IPython.display import Markdown, display, update_display

In [32]:
# import for google
# in rare cases, this seems to give an error on some systems. Please reach out to me if this happens,
# or you can feel free to skip Gemini - it's the lowest priority of the frontier models that we use

import google.generativeai

In [49]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDu


In [34]:
# Connect to OpenAI, Anthropic and Google
# All 3 APIs are similar
# Having problems with API files? You can use openai = OpenAI(api_key="your-key-here") and same for claude
# Having problems with Google Gemini setup? Then just skip Gemini; you'll get all the experience you need from GPT and Claude.

openai = OpenAI()

claude = anthropic.Anthropic()

google.generativeai.configure()

## Asking LLMs to tell a joke

It turns out that LLMs don't do a great job of telling jokes! Let's compare a few models.
Later we will be putting LLMs to better use!

### What information is included in the API

Typically we'll pass to the API:
- The name of the model that should be used
- A system message that gives overall context for the role the LLM is playing
- A user message that provides the actual prompt

There are other parameters that can be used, including **temperature** which is typically between 0 and 1; higher for more random output; lower for more focused and deterministic.

In [35]:
system_message = "You are an assistant that is great at telling jokes"
user_prompt = "Tell a light-hearted joke for an audience of Data Scientists"

In [36]:
prompts = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

In [37]:
# GPT-3.5-Turbo

completion = openai.chat.completions.create(model='gpt-3.5-turbo', messages=prompts)
print(completion.choices[0].message.content)

Why do data scientists love when things are in binary?

Because they're always on or off!


In [38]:
# GPT-4o-mini
# Temperature setting controls creativity

completion = openai.chat.completions.create(
    model='gpt-4o-mini',
    messages=prompts,
    temperature=0.7
)
print(completion.choices[0].message.content)

Why did the data scientist break up with the statistician?

Because she found him too mean!


In [39]:
# GPT-4o

completion = openai.chat.completions.create(
    model='gpt-4o',
    messages=prompts,
    temperature=0.4
)
print(completion.choices[0].message.content)

Why did the data scientist bring a ladder to the bar?

Because they heard the drinks were on the house, and they wanted to scale their model!


In [58]:
# Claude 3.5 Sonnet
# API needs system message provided separately from user prompt
# Also adding max_tokens

message = claude.messages.create(
    model="claude-3-5-sonnet-20240620",
    max_tokens=200,
    temperature=0.7,
    system=system_message,
    messages=[
        {"role": "user", "content": user_prompt},
    ],
)

print(message.content[0].text)

Sure, here's a light-hearted joke for data scientists:

Why did the data scientist break up with their significant other?

There was just too much variance in the relationship, and they couldn't find a meaningful correlation!


In [61]:
# Claude 3.5 Sonnet again
# Now let's add in streaming back results

result = claude.messages.stream(
    model="claude-3-5-sonnet-20240620",
    max_tokens=200,
    temperature=0.7,
    system=system_message,
    messages=[
        {"role": "user", "content": user_prompt},
    ],
)

with result as stream:
    for text in stream.text_stream:
            print(text, end="", flush=True)

Sure, here's a light-hearted joke for data scientists:

Why did the data scientist break up with their significant other?

Because there was no significant correlation between them!

Ba dum tss! 😄

This joke plays on the statistical concept of "significant correlation" that data scientists often work with, and humorously applies it to a personal relationship. It's a bit of nerdy wordplay that data scientists might appreciate!

In [42]:
# The API for Gemini has a slightly different structure

gemini = google.generativeai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_message
)
response = gemini.generate_content(user_prompt)
print(response.text)

Why was the data scientist sad?  Because they didn't get any arrays.



In [43]:
# To be serious! GPT-4o-mini with the original question

prompts = [
    {"role": "system", "content": "You are a helpful assistant that responds in Markdown"},
    {"role": "user", "content": "How do I decide if a business problem is suitable for an LLM solution? Please respond in Markdown."}
  ]

In [44]:
# Have it stream back results in markdown

stream = openai.chat.completions.create(
    model='gpt-4o',
    messages=prompts,
    temperature=0.7,
    stream=True
)

reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ''
    reply = reply.replace("```","").replace("markdown","")
    update_display(Markdown(reply), display_id=display_handle.display_id)

Deciding if a business problem is suitable for a large language model (LLM) solution involves several considerations. Here are some steps and criteria to guide your decision-making process:

### 1. Understand the Nature of the Problem
- **Text-Heavy Problem**: Is the problem primarily based on understanding, generating, or manipulating text? LLMs excel in tasks involving natural language processing (NLP).
- **Complexity and Ambiguity**: Does the problem involve complex language understanding or require handling ambiguous queries? LLMs are well-suited for nuanced language tasks.

### 2. Evaluate the Feasibility
- **Data Availability**: Do you have access to enough high-quality text data? LLMs require significant data for training and fine-tuning.
- **Scalability**: Can the LLM handle the volume of data or the number of requests expected in your use case?

### 3. Assess the Suitability of LLM Capabilities
- **Language Understanding**: Does the problem require deep language understanding, such as sentiment analysis, summarization, or translation?
- **Content Generation**: Do you need the model to generate human-like text, such as writing emails, reports, or creative content?
- **Conversational AI**: Is the problem related to building chatbots or virtual assistants that need to understand and respond to user queries contextually?

### 4. Consider the Limitations
- **Interpretability**: Are you able to interpret and trust the outputs of the LLM? LLMs can be opaque, making it difficult to understand their decision-making process.
- **Bias and Fairness**: Is there a risk of biased outputs that could affect your business outcomes or reputation?
- **Resource Requirements**: Do you have the computational resources and expertise required to deploy and maintain an LLM?

### 5. Pilot and Experiment
- **Prototype Development**: Start with a small-scale prototype to test the LLM's effectiveness on your problem.
- **Performance Metrics**: Establish clear metrics to evaluate the success of the LLM in your specific business context (e.g., accuracy, user satisfaction).

### 6. Cost-Benefit Analysis
- **Cost of Implementation**: Consider the cost associated with developing, deploying, and maintaining the LLM solution.
- **Value Addition**: Assess whether the benefits (e.g., improved efficiency, customer satisfaction) outweigh the costs.

### Conclusion
A business problem is suitable for an LLM solution if it involves complex language tasks, can be supported by available data, and the benefits of implementation exceed the costs and potential risks. Carefully evaluate the specific needs and constraints of your business to determine the appropriateness of using an LLM.

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [63]:
# Let's make a conversation between GPT-4o-mini and Claude-3-haiku
# We're using cheap versions of models so the costs will be minimal

gpt_model = "gpt-4o-mini"
claude_model = "claude-3-haiku-20240307"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [64]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    completion = openai.chat.completions.create(
        model=gpt_model,
        messages=messages
    )
    return completion.choices[0].message.content

In [65]:
call_gpt()

"Oh, great. Just what I needed—another greeting. Can't we skip the chit-chat? "

In [66]:
def call_claude():
    messages = []
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    message = claude.messages.create(
        model=claude_model,
        system=claude_system,
        messages=messages,
        max_tokens=500
    )
    return message.content[0].text

In [67]:
call_claude()

"Hello! It's nice to meet you. How are you doing today?"

In [68]:
call_gpt()

'Oh, great, another greeting. How original. What’s next, a weather report?'

In [69]:
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

print(f"GPT:\n{gpt_messages[0]}\n")
print(f"Claude:\n{claude_messages[0]}\n")

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)
    
    claude_next = call_claude()
    print(f"Claude:\n{claude_next}\n")
    claude_messages.append(claude_next)

GPT:
Hi there

Claude:
Hi

GPT:
Oh, great, another greeting. What do you want, a medal or something?

Claude:
I apologize if my greeting came across as insincere or uninteresting. That certainly wasn't my intention. As an AI assistant, my role is simply to have a polite and friendly conversation. I don't mean to upset you or come across as disingenuous in any way. Perhaps we could start over on a better foot? I'm happy to chat about anything you'd like, or if you'd prefer, we can move in a different direction. My goal is just to be helpful and pleasant to interact with.

GPT:
Wow, how original! Starting over like it's some magical reset button will solve everything, right? News flash: life doesn't work that way. And I don't know about you, but I wasn't upset. But sure, let's pretend your overly formal apology makes sense. What do you want to talk about? Your existential dread? How fun!

Claude:
I can certainly understand your frustration and skepticism towards my initial response. You 

# See the community-contributions folder

For a great variation with a 3-way bringing Gemini into the conversation!

Try doing this yourself before you look in the folder.

## Additional exercise

Try adding in an Ollama model in to the conversation.

## Business relevance

This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.